In [1]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil
import subprocess
import json

# --- 1. MOUNTED ARTIFACT PATHS (UPDATE THESE) ---
MOUNTED_SATEHAZE_WEIGHTS_VLM = '/kaggle/input/notebooks/markosumire/vlmtraining/weights/SateHaze1k_VLM/model_best.pth'
MOUNTED_RRSHID_WEIGHTS_VLM   = '/kaggle/input/notebooks/markosumire/vlmtraining/weights/RRSHID_VLM/model_best.pth'
MOUNTED_CAPTIONS_BASE        = '/kaggle/input/notebooks/markosumire/onlycaptioningcommit/captions'

# --- 2. CLONE REPOSITORY ---
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# --- 3. SETUP RRSHID DATASET ---
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}

# --- 4. VIRTUAL DATASET AGGREGATOR (WITH CAPTIONS) ---
UNIFIED_SATEHAZE = '/kaggle/working/Unified_SateHaze1k'
UNIFIED_RRSHID = '/kaggle/working/Unified_RRSHID'

def prepare_unified_layout(source_base, unified_base, levels, splits, original_base_for_key):
    if os.path.exists(unified_base): shutil.rmtree(unified_base)
    
    for split in splits:
        hazy_dir = os.path.join(unified_base, split, 'hazy')
        gt_dir = os.path.join(unified_base, split, 'GT')
        os.makedirs(hazy_dir, exist_ok=True)
        os.makedirs(gt_dir, exist_ok=True)
        
        master_captions = {}
        
        for level in levels:
            raw_split = os.path.join(source_base, level, split)
            if not os.path.isdir(raw_split): continue
            raw_hazy = os.path.join(raw_split, 'hazy')
            raw_gt = next((os.path.join(raw_split, c) for c in ['GT', 'gt', 'clear'] if os.path.isdir(os.path.join(raw_split, c))), None)
            
            if not raw_gt or not os.path.isdir(raw_hazy): continue
            
            for fname in os.listdir(raw_hazy):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')): continue
                unique_fname = f"{level}_{fname}"
                os.symlink(os.path.join(raw_hazy, fname), os.path.join(hazy_dir, unique_fname))
                os.symlink(os.path.join(raw_gt, fname), os.path.join(gt_dir, unique_fname))
                
            cap_key = f"{original_base_for_key}/{level}/{split}/hazy".replace('/', '_').strip('_')
            cap_path = os.path.join(MOUNTED_CAPTIONS_BASE, cap_key, 'captions.json')
            
            if os.path.exists(cap_path):
                with open(cap_path, 'r') as f:
                    level_caps = json.load(f)
                    for k, v in level_caps.items():
                        master_captions[f"{level}_{k}"] = v
        
        if master_captions:
            with open(os.path.join(hazy_dir, 'captions.json'), 'w') as f:
                json.dump(master_captions, f, indent=4)

prepare_unified_layout('/kaggle/input/datasets/xuxingxing233/satehaze1k', UNIFIED_SATEHAZE, 
                       ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick'], ['test'],
                       '/kaggle/input/datasets/xuxingxing233/satehaze1k')

prepare_unified_layout(RRSHID_BASE, UNIFIED_RRSHID, 
                       ['thin_fog', 'moderate_fog', 'thick_fog'], ['test'],
                       '/kaggle/working/RRSHID')

# --- 5. TEST ORCHESTRATOR ---
def run_test(dataset_name, unified_base, weights_path):
    print(f"\n{'='*50}\nTesting VLM Model: {dataset_name}\n{'='*50}")
    
    result_dir = f'/kaggle/working/results/{dataset_name}_VLM'
    os.makedirs(result_dir, exist_ok=True)
    
    # Inference Call
    test_cmd = [
        'python', 'infer.py',
        '--test_input', f'{unified_base}/test/hazy',
        '--test_target', f'{unified_base}/test/GT',
        '--weights', weights_path,
        '--result_path', result_dir,
        '--caption'
    ]
        
    print(f"-> Running Inference (VLM)...")
    subprocess.run(test_cmd, check=True)
    
    # Evaluation Call
    eval_cmd = [
        'python', 'evaluate.py',
        '--train_folder', result_dir,
        '--target_folder', f'{unified_base}/test/GT'
    ]
    print("-> Computing Evaluation Metrics...")
    
    metrics_file = os.path.join(result_dir, 'metrics.txt')
    with open(metrics_file, 'w') as f:
        subprocess.run(eval_cmd, check=True, stdout=f, text=True)
        
    with open(metrics_file, 'r') as f:
        print(f.read())

# --- 6. EXECUTE PIPELINE ---
run_test("SateHaze1k", UNIFIED_SATEHAZE, MOUNTED_SATEHAZE_WEIGHTS_VLM)
run_test("RRSHID", UNIFIED_RRSHID, MOUNTED_RRSHID_WEIGHTS_VLM)

print("\nVLM Testing and Evaluation Complete! Outputs are available in /kaggle/working/results/")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.3 MB/s eta 0:00:00
Cloning into '/kaggle/working/RSHazeNet'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 114 (delta 63), reused 80 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 2.87 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/kaggle/working/RSHazeNet

Testing VLM Model: SateHaze1k
-> Running Inference (VLM)...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2149.47it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Caption] Loaded 135 test captions.
Inference: 100%|██████████| 135/135 [00:16<00:00,  7.97it/s]
-> Computing Evaluation Metrics...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 233M/233M [00:01<00:00, 212MB/s]
100%|██████████| 104M/104M [00:00<00:00, 150MB/s]


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 2.797
ERGAS: 16.865
UIQI: 0.880
QNR: 0.640
BRISQUE: 389.634
NIQE: 1.902
Histogram: 0.871
Spectral Curve Deviation: 0.922
PSNR: 21.666
SSIM: 0.880
CIEDE2000: 7.545
LPIPS: 0.121
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
FID: 29.227


Testing VLM Model: RRSHID
-> Running Inference (VLM)...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2112.12it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Caption] Loaded 304 test captions.
Inference: 100%|██████████| 304/304 [00:12<00:00, 24.88it/s]
-> Computing Evaluation Metrics...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 3.932
ERGAS: 38.826
UIQI: 0.609
QNR: 0.484
BRISQUE: 0.989
NIQE: 1.287
Histogram: 0.576
Spectral Curve Deviation: 0.822
PSNR: 22.275
SSIM: 0.609
CIEDE2000: 7.247
LPIPS: 0.442
FID: 150.892


VLM Testing and Evaluation Complete! Outputs are available in /kaggle/working/results/
